### Complete Chatting RAG-VLM Notebook Code for Kaggle

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "" # This makes PyTorch "invisible" to any GPUs

In [2]:
# ==========================================
# CELL 1: Environment & Dependency Setup
# ==========================================
!pip install -q "pillow<11.1.0"
!pip install -q -U transformers accelerate bitsandbytes datasets faiss-cpu langchain langchain-community langchain-huggingface sentence-transformers

import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("Classic")
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 58.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
# ==========================================
# CELL 2: Load Instruction-Tuned MedGemma Safely
# ==========================================
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# Use the instruction-tuned chat variant (-it)
model_id = "google/medgemma-1.5-4b-it"

print(f"Loading processor and conversational model: {model_id}...")
processor = AutoProcessor.from_pretrained(model_id, token=hf_token)

# Load safely on CPU first to avoid hardware kernel init crashes, then shift to GPU
print("Loading model weights onto CPU memory first...")
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    device_map=None,
    token=hf_token
)

device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    if device == "cuda":
        print("Shifting conversational model to CUDA...")
        model = model.to(torch.bfloat16).to("cuda")
        print("Model successfully moved to GPU!")
    else:
        print("Using CPU mode.")
except Exception as e:
    print(f"GPU shift skipped ({e}). Staying on CPU.")
    model = model.to("cpu")

print("MedGemma Chat Assistant is ready!")

Loading processor and conversational model: google/medgemma-1.5-4b-it...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Using CPU mode.
MedGemma Chat Assistant is ready!


In [4]:
# ==========================================
# CELL 3: Setup RAG Knowledge Base (CPU Embeddings)
# ==========================================
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

medical_knowledge_base = [
    Document(page_content="Gliomas are primary brain tumors originating from glial cells. High-grade gliomas typically present on T1-weighted contrast-enhanced MRIs with irregular ring enhancement, central necrosis, and surrounding vasogenic edema."),
    Document(page_content="Meningiomas are typically benign, slow-growing extra-axial tumors originating from the meninges. On brain MRI scans, they commonly appear as well-circumscribed, dural-attached masses that show intense, uniform contrast enhancement."),
    Document(page_content="Pituitary adenomas develop in the pituitary gland inside the sella turcica. Macro-adenomas can compress the optic chiasm, leading to visual field defects.")
]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vectorstore = FAISS.from_documents(medical_knowledge_base, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
print("RAG Knowledge Vector Store Index Loaded.")

/tmp/ipykernel_23/576050081.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG Knowledge Vector Store Index Loaded.


In [5]:
# ==========================================
# CELL 4: Multi-Turn Conversational RAG Engine
# ==========================================
from PIL import Image

# Initialize chat memory structure with a medical persona guardrail
chat_history = [
    {
        "role": "system",
        "content": (
            "You are a compassionate, clinical neuro-oncology assistant chatbot. "
            "Your job is to converse with patients and answer questions accurately. "
            "You MUST strictly base your medical explanations on the provided RAG clinical context "
            "and visual scan analysis. Never hallucinate facts or give false diagnoses."
        )
    }
]



In [6]:
# ==========================================
# CELL: Force CPU Inference Engine
# ==========================================
def chat_with_patient(image_path, user_message):
    # 1. RAG Retrieval
    retrieved_docs = retriever.invoke(user_message)
    rag_context = retrieved_docs[0].page_content if retrieved_docs else "General neuro-oncology knowledge."
    
    enhanced_message = (
        f"[Verified Clinical Context - Strict Fact Guide]: {rag_context}\n\n"
        f"[Patient/User Message]: {user_message}"
    )
    
    # 2. Attach image
    if image_path and os.path.exists(image_path):
        image = Image.open(image_path).convert("RGB")
        content_payload = [{"type": "image", "image": image}, {"type": "text", "text": enhanced_message}]
    else:
        content_payload = [{"type": "text", "text": enhanced_message}]
        
    chat_history.append({"role": "user", "content": content_payload})
    
    # 3. Apply Chat Template
    prompt_text = processor.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
    
    active_images = [
        item["image"] for turn in chat_history if isinstance(turn["content"], list) 
        for item in turn["content"] if item.get("type") == "image"
    ]
    
    # --- CRITICAL FIX: Ensure inputs are forced to CPU ---
    inputs = processor(
        text=[prompt_text], 
        images=active_images if active_images else None, 
        return_tensors="pt"
    ).to("cpu") 
    
    # --- CRITICAL FIX: Ensure model is on CPU for generation ---
    model.to("cpu")
    
    # 4. Generate conversational output strictly on CPU
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=250, 
            do_sample=True, 
            temperature=0.2
        )
        
    input_token_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_token_length:]
    response_text = processor.decode(generated_tokens, skip_special_tokens=True)
    
    chat_history.append({"role": "assistant", "content": [{"type": "text", "text": response_text}]})
    
    return response_text

---

### How to Use It for Back-and-Forth Chatting

Now you can test a live back-and-forth session where the model maintains context across multiple questions:

In [7]:
# --- Turn 1: Patient uploads an MRI scan and asks an initial question ---
sample_img_path = "/kaggle/input/datasets/nahin333/brain-tumor-dataset/Meningioma/image_1.jpg" # Adjust to your path

print("Patient: Hello, can you review this scan and tell me what kind of tumor this looks like?")
reply_1 = chat_with_patient(sample_img_path, "Hello, can you review this scan and tell me what kind of tumor this looks like?")
print(f"\nAssistant: {reply_1}\n" + "-"*50)

# --- Turn 2: Patient asks a follow-up question without re-uploading the image ---
print("Patient: What are the main visual features of this condition?")
reply_2 = chat_with_patient(None, "What are the main visual features of this condition?")
print(f"\nAssistant: {reply_2}\n" + "-"*50)

# --- Turn 3: Patient asks another follow-up question ---
print("Patient: Is it normally considered benign or malignant?")
reply_3 = chat_with_patient(None, "Is it normally considered benign or malignant?")
print(f"\nAssistant: {reply_3}\n" + "-"*50)

Patient: Hello, can you review this scan and tell me what kind of tumor this looks like?

Assistant: Based on the provided clinical context, the scan shows a well-circumscribed, dural-attached mass that enhances intensely and uniformly. This appearance is characteristic of a meningioma.

It's important to remember that I am an AI and cannot provide a definitive diagnosis. A qualified medical professional needs to interpret the scan in the context of the patient's full medical history and perform further investigations to confirm the diagnosis and determine the best course of treatment.
--------------------------------------------------
Patient: What are the main visual features of this condition?

Assistant: Based on the provided clinical context, pituitary adenomas, especially macro-adenomas, can cause visual field defects. These often occur because the tumor can compress the optic chiasm, which is located just above the pituitary gland. This compression can lead to specific visual di

In [8]:
!pip install -q bert-score rouge-score evaluate

from bert_score import score as bert_score
from rouge_score import rouge_scorer

# 1. Prepare your data
# candidates: the AI model's answers
# references: the ground-truth "Gold Standard" medical answers
candidates = ["The MRI shows a benign meningioma with dural attachment."]
references = ["This is a meningioma, a benign extra-axial tumor."]



# Use the 'rescale_with_baseline=False' and 'device="cpu"' flags 
# to prevent the library from attempting to trigger CUDA kernels.
P, R, F1 = bert_score(
    candidates, 
    references, 
    lang="en", 
    verbose=True, 
    device="cpu"  # <--- CRITICAL FIX: Forces calculation on RAM/CPU
)

print(f"BERTScore F1 (Semantic Match): {F1.mean().item():.4f}")
print(f"BERTScore F1 (Semantic Match): {F1.mean().item():.4f}")

# 3. Calculate ROUGE (Keyword/Content overlap)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
scores = scorer.score(references[0], candidates[0])
print(f"ROUGE-1 Precision: {scores['rouge1'].precision:.4f}")
print(f"ROUGE-L (Longest Match): {scores['rougeL'].fmeasure:.4f}")

# 4. LLM-as-a-Judge (Conceptual implementation)
def evaluate_factuality(ai_response, rag_context):
    """
    In a real scenario, send the AI response + retrieved RAG documents 
    to a powerful model with this prompt:
    'Is the following response supported by the provided medical context? 
    Answer YES/NO and explain if there are any hallucinations.'
    """
    return "This is where you integrate GPT-4o or another Judge LLM."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.33 seconds, 3.05 sentences/sec
BERTScore F1 (Semantic Match): 0.9170
BERTScore F1 (Semantic Match): 0.9170
ROUGE-1 Precision: 0.3333
ROUGE-L (Longest Match): 0.2222
